In [1]:
import pandas as pd
import numpy as np
import time
import sys
sys.path.append('..')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from cuml.ensemble import RandomForestRegressor as cuRF
print('All imports OK')

All imports OK


In [2]:
df = pd.read_csv(r'../output_csv/HCMUT-SuperNodeXP-2017-1.0.swf.csv')
print(f'Raw dataset shape: {df.shape}')

# Filter successful jobs with runtime >= 600s
df = df[df['status'] == 1]
df = df[df['run_time'] >= 600]
print(f'After filter: {df.shape}')

Raw dataset shape: (15886, 18)
After filter: (10131, 18)


In [3]:
# === Feature Engineering ===
# 1. Use ALL informative features (drop near-zero correlation ones)
# 2. Add engineered features
# 3. Log-transform target (skewness 5.93 -> 0.07)

df_clean = df.copy()

# Cyclic encoding for submit_time
hours = (df_clean['submit_time'] % 86400) / 3600.0
df_clean['submit_sin'] = np.sin(2 * np.pi * hours / 24.0)
df_clean['submit_cos'] = np.cos(2 * np.pi * hours / 24.0)

# Day of week (cyclic)
day_of_week = (df_clean['submit_time'] % (86400 * 7)) / (86400 * 7)
df_clean['dow_sin'] = np.sin(2 * np.pi * day_of_week)
df_clean['dow_cos'] = np.cos(2 * np.pi * day_of_week)

# Log transform target
df_clean['log_run_time'] = np.log1p(df_clean['run_time'])

# Feature columns — use strongest features, drop noisy ones
feature_columns = [
    'requested_processors',
    'num_allocated_processors',
    'avg_cpu_time_used',
    'used_memory',
    'wait_time',
    'user_id',
    'group_id',
    'executable_id',
    'submit_sin',
    'submit_cos',
    'dow_sin',
    'dow_cos',
]

target_column = 'log_run_time'  # LOG TRANSFORM!

print(f'Features ({len(feature_columns)}): {feature_columns}')
print(f'Target: {target_column} (log-transformed)')
print(f'Target skewness: run_time={df["run_time"].skew():.2f}, log_run_time={df_clean["log_run_time"].skew():.2f}')

Features (12): ['requested_processors', 'num_allocated_processors', 'avg_cpu_time_used', 'used_memory', 'wait_time', 'user_id', 'group_id', 'executable_id', 'submit_sin', 'submit_cos', 'dow_sin', 'dow_cos']
Target: log_run_time (log-transformed)
Target skewness: run_time=5.93, log_run_time=0.07


In [4]:
# Train Random Forest (GPU - cuML) with improved features
rmse_lst, mae_lst, r2_lst, infer_time_lst = [], [], [], []

for iter in range(5):
    print(f"Iteration {iter + 1} / 5")
    
    # Split data
    X = df_clean[feature_columns].values
    y = df_clean[target_column].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
    
    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # Train cuML Random Forest on GPU
    rf_model = cuRF(
        n_estimators=200,
        max_depth=20,
        min_samples_split=2,
        max_features=1.0,
        bootstrap=True
    )
    rf_model.fit(X_train_s, y_train)
    
    # Predict
    start_time = time.time()
    y_pred_log = rf_model.predict(X_test_s)
    end_time = time.time()
    
    # Inverse log transform to get original scale
    Y_pred = np.expm1(y_pred_log)
    Y_true = np.expm1(y_test)
    
    rmse = np.sqrt(mean_squared_error(Y_true, Y_pred))
    mae = mean_absolute_error(Y_true, Y_pred)
    r2 = r2_score(Y_true, Y_pred)
    infer_time = (end_time - start_time) / len(y_test)
    
    rmse_lst.append(rmse)
    mae_lst.append(mae)
    r2_lst.append(r2)
    infer_time_lst.append(infer_time)

Iteration 1 / 5
Iteration 2 / 5
Iteration 3 / 5
Iteration 4 / 5
Iteration 5 / 5


In [5]:
# Save results
results_df = pd.DataFrame({
    'RMSE': rmse_lst,
    'MAE': mae_lst,
    'R2': r2_lst,
    'Inference Time': infer_time_lst
})
results_df.to_csv('../output_HCMUT/RandomForest_results.csv', index=False)

print(f"\n{'='*60}")
print(f"Random Forest (GPU - cuML) — HCMUT Dataset (Improved)")
print(f"{'='*60}")
print(f"Mean RMSE: {np.mean(rmse_lst):.4f} ± {np.std(rmse_lst):.4f}")
print(f"Mean MAE:  {np.mean(mae_lst):.4f} ± {np.std(mae_lst):.4f}")
print(f"Mean R²:   {np.mean(r2_lst):.4f} ± {np.std(r2_lst):.4f}")
print(f"Mean Inference Time: {np.mean(infer_time_lst):.6f}s")
print(f"\nImprovements applied:")
print(f"  - Log transform target (skewness 5.93 -> 0.07)")
print(f"  - Added: avg_cpu_time_used, num_allocated_processors, group_id, executable_id")
print(f"  - Added: cyclic day-of-week features")
print(f"  - Removed: requested_time (corr=0.015, pure noise)")
print(f"  - Increased: n_estimators=200, max_depth=20")


Random Forest (GPU - cuML) — HCMUT Dataset (Improved)
Mean RMSE: 49883.5525 ± 9855.8874
Mean MAE:  10762.8482 ± 1201.6609
Mean R²:   0.9043 ± 0.0305
Mean Inference Time: 0.000273s

Improvements applied:
  - Log transform target (skewness 5.93 -> 0.07)
  - Added: avg_cpu_time_used, num_allocated_processors, group_id, executable_id
  - Added: cyclic day-of-week features
  - Removed: requested_time (corr=0.015, pure noise)
  - Increased: n_estimators=200, max_depth=20
